In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/Domain-SupportSpecialist'
print('Project root:', PROJECT)


Mounted at /content/drive
Project root: /content/drive/MyDrive/Domain-SupportSpecialist


In [2]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU -- Runtime -> Change runtime type -> T4 GPU -> Restart session")
print("GPU OK:", torch.cuda.get_device_name(0))


GPU OK: Tesla T4


In [3]:
%%capture
!pip install --no-deps unsloth
!pip install unsloth_zoo
!pip install --no-deps peft accelerate bitsandbytes xformers


In [4]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"{PROJECT}/adapters/stage1_non_instruction",
    max_seq_length = 2048, dtype = None, load_in_4bit = True,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
QUESTIONS = [
    "How can I cancel my order after it has been placed?",
    "My package says delivered but I never received it, what do I do?",
    "How long does a refund take to appear on my card?",
    "Can I change the delivery address after checkout?",
    "The item I received is damaged, what are my options?",
    "How do I track my order status?",
    "I was charged twice for one order, how do I fix this?",
    "Can I get a replacement instead of a refund?",
    "What happens if I miss the delivery attempt?",
    "How do I apply a discount code after placing an order?",
]
def ask(prompt):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True,
                add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)


In [6]:
import json
from datasets import Dataset
rows = [json.loads(l) for l in open(f"{PROJECT}/data/instruction_dataset.jsonl")]
def to_text(r):
    messages = [{"role": "user", "content": r["instruction"]}, {"role": "assistant", "content": r["response"]}]
    return tokenizer.apply_chat_template(messages, tokenize=False)
sft_ds = Dataset.from_dict({"text": [to_text(r) for r in rows]})
print(len(sft_ds), "training examples")


200 training examples


In [7]:
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
)
from trl import SFTTrainer
from transformers import TrainingArguments
trainer = SFTTrainer(
    model = model, processing_class = tokenizer, train_dataset = sft_ds,
    dataset_text_field = "text", max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2, gradient_accumulation_steps = 4,
        warmup_steps = 10, num_train_epochs = 3,
        learning_rate = 2e-4, fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), logging_steps = 10,
        output_dir = "stage2_out", optim = "adamw_8bit", save_strategy = "no",
    ),
)
trainer.train()
import os
os.makedirs(f"{PROJECT}/adapters/stage2_sft", exist_ok=True)
model.save_pretrained(f"{PROJECT}/adapters/stage2_sft")
tokenizer.save_pretrained(f"{PROJECT}/adapters/stage2_sft")


Unsloth: Already have LoRA adapters! We shall skip this step.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 3 | Total steps = 75
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,1.576857
20,0.987686
30,0.887399
40,0.795725
50,0.741381
60,0.674428
70,0.648318


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage2_sft/tokenizer_config.json.


('/content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage2_sft/tokenizer_config.json',
 '/content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage2_sft/chat_template.jinja',
 '/content/drive/MyDrive/Domain-SupportSpecialist/adapters/stage2_sft/tokenizer.json')

In [8]:
FastLanguageModel.for_inference(model)
sft_answers = {q: ask(q) for q in QUESTIONS}
for q, a in sft_answers.items():
    print("Q:", q, "\nA:", a, "\n---")


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

Q: How can I cancel my order after it has been placed? 
A: I understand your need to cancel your order and we're here to assist you with that process! To initiate the cancellation of your order #SC-20371, please follow these steps:

1. Access Your Account: Log in to your account portal at support.example.com using your credentials.
2. Navigate to Order History: Once logged in, go to the "Order History" or "Order History" section.
3. Locate the Specific Order: Look for the order associated with the number #SC-25968 and click on it to view more details.
4. Initiate Cancellation: Within the order details page, find the option labeled "Order History" and select it.
5. Confirm Cancellation: Follow any additional prompts or questions that may be displayed during the cancellation process.

If you encounter any difficulties or have further questions throughout this process, our dedicated customer support team is available Monday-Friday, 9 AM-6 PM EST at 1-80 
---
Q: My package says delivered b

In [9]:
with open(f'{PROJECT}/reports/base_model_evaluation.md', encoding='utf-8') as f:
    base_report = f.read()
base_answers = {}
for line in base_report.splitlines():
    if line.startswith('|') and 'Base Model Answer' not in line and '---' not in line:
        parts = [c.strip() for c in line.strip('|').split('|')]
        if len(parts) >= 2:
            base_answers[parts[0]] = parts[1]

rows_md = "\n".join(
    f"| {q} | {base_answers.get(q, '_(see report)_')} | {sft_answers[q].strip()} | _(fill in)_ | _(fill in)_ |"
    for q in QUESTIONS
)
report = f"""# Base vs SFT Model Comparison

| Question | Base | SFT | Which is Better? | Reason |
|---|---|---|---|---|
{rows_md}
"""
with open(f'{PROJECT}/reports/sft_model_comparison.md', 'w', encoding='utf-8') as f:
    f.write(report)
print("Report written.")


Report written.
